<img src="http://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0">

# Framework Classes and Functions

This section explains the usage of some basic framework classes and functions of DX Analytics. Mainly some helper functions, the discounting classes and the market environment class used to store market data and other parameters/data needed to model, value and risk manage derivative instruments.

In [ ]:
!git clone https://github.com/tpq-classes/computational_finance_dx.git
import sys
sys.path.append('computational_finance_dx')

!git clone https://github.com/yhilpisch/dx.git
sys.path.append('dx')

In [ ]:
from dx import *

In [ ]:
np.set_printoptions(precision=3)

## Helper Functions

There are two helper functions used regulary:

* <code>get_year_deltas</code>: get a list of year deltas (decimal fractions) relative to first value in time_list
* <code>sn_random_numbers</code>: get an array of standard normally distributed pseudo-random numbers

### get_year_deltas

Suppose we have a `list` object containing a number of `datetime` objects.

In [ ]:
time_list = [dt.datetime(2020, 1, 1),
             dt.datetime(2020, 4, 1),
             dt.datetime(2020, 6, 15),
             dt.datetime(2020, 10, 21)]

Passing this object to the `get_year_deltas` functions yields a list of year fractions representing the time intervals between the dates given. This is sometimes used e.g. for discounting purposes.

In [ ]:
get_year_deltas(time_list)

### sn_random_numbers

Monte Carlo simulation of course relies heavily an the use of random numbers. The function `sn_random_numbers` is a wrapper function around the pseudo-random number generator of the `NumPy` library. It implements antithetic variates and moment matching as generic variance reduction techniques. It also allows to fix the seed value for the random number generator. The `shape` parameter is a `tuple` object of three integers.

In [ ]:
ran = sn_random_numbers((2, 3, 4), antithetic=True,
                        moment_matching=True, fixed_seed=False)

In [ ]:
ran

Using moment matching makes sure that the first and second moments match exactly 0 and 1, respectively.

In [ ]:
ran.mean()

In [ ]:
ran.std()

Setting the first value of the `shape` parameter to 1 yields a two-dimensional `ndarray` object.

In [ ]:
ran = sn_random_numbers((1, 3, 4), antithetic=True,
                        moment_matching=True, fixed_seed=False)

In [ ]:
ran

## Discounting Classes

In the risk-neutral valuation of derivative instrumente, discounting payoffs is a major task. The following discounting classes are implemented:

* `constant_short_rate`: fixed short rate
* `deterministic_yield`: deterministic yiels/term structure

### constant_short_rate

The `constant_short_rate` class represents the most simple case for risk-neutral discounting. A discounting object is defined by instatiating the class and providing a name and a decimal short rate value only.

In [ ]:
r = constant_short_rate('r', 0.05)

In [ ]:
r.name

In [ ]:
r.short_rate

The object has a method `get_forward_rates` to generate forward rates given, for instance, a `list` object of `datetime` objects.

In [ ]:
r.get_forward_rates(time_list)

Similarly, the method `get_discount_factors` returns discount factors for such a `list` object.

In [ ]:
r.get_discount_factors(time_list)

You can also pass, for instance, an `ndarry` object containing year fractions.

In [ ]:
r.get_discount_factors(np.array([0., 1., 1.5, 2.]),
                       dtobjects=False)

### deterministic_short_rate

The `deterministic_short_rate` class allows to model an interest rate term structure. To this end, you need to pass a `list` object of `datetime` and yield pairs to the  class.

In [ ]:
yields = [(dt.datetime(2020, 1, 1), 0.02),
          (dt.datetime(2020, 3, 1), 0.03),
          (dt.datetime(2020, 10, 15), 0.035),
          (dt.datetime(2020, 12, 31), 0.04)]

In [ ]:
y = deterministic_short_rate('y', yields)

In [ ]:
y.name

In [ ]:
y.yield_list

The method `get_interpolated_yields` implements an interpolation of the yield data and returns the interpolated yields given a `list` object of `datetime` objects.

In [ ]:
y.get_interpolated_yields(time_list)

In similar fashion, the methods `get_forward_rates` and `get_discount_factors` return forward rates and discount factors, respcectively.

In [ ]:
y.get_forward_rates(time_list)

In [ ]:
y.get_discount_factors(time_list)

## Market Environment

The `market_environment` class is used to collect relevant data for the modeling, valuation and risk management of single derivatives instruments and portfolios composed of such instruments. A `market_environment` object stores:

* `constants`: e.g. maturity date of option
* `lists`: e.g. list of dates
* `curves`: e.g. discounting objects

A `market_environment` object is instantiated by providing a name as a `string` object and the pricing date as a `datetime` object.

In [ ]:
me = market_environment(name='me', pricing_date=dt.datetime(2020, 1, 1))

In [ ]:
me.constants

Constants are added via the `add_constant` method and providing a key and the value.

In [ ]:
me.add_constant('initial_value', 100.)

In [ ]:
me.add_constant('volatility', 0.25)

In [ ]:
me.constants

Lists of data are added via the  `add_list` method.

In [ ]:
me.add_list('dates', time_list)

The `add_curve` method does the same for curves.

In [ ]:
me.add_curve('discount_curve_1', r)

In [ ]:
me.add_curve('discount_curve_2', y)

The single data objects are stored in separate `dictionary` objects.

In [ ]:
me.constants

In [ ]:
me.lists

In [ ]:
me.curves

Data is retrieved from a `market_environment` object via the `get_constant`, `get_list` and `get_curve` methods and providing the respective key.

In [ ]:
me.get_constant('volatility')

In [ ]:
me.get_list('dates')

In [ ]:
me.get_curve('discount_curve_1')

Retrieving, for instance, a discounting object you can in one step retrieve it and call a method on it.

In [ ]:
me.get_curve('discount_curve_2').get_discount_factors(time_list)

**Copyright, License & Disclaimer**

© Dr. Yves J. Hilpisch | The Python Quants GmbH

DX Analytics (the "dx library" or "dx package") is licensed under the GNU Affero General
Public License version 3 or later (see http://www.gnu.org/licenses/).

DX Analytics comes with no representations or warranties, to the extent
permitted by applicable law.

http://tpq.io | [dx@tpq.io](mailto:team@tpq.io) |
http://twitter.com/dyjh

<img src="http://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>

**Quant Platform** | http://pqp.io

**Python for Finance Training** | http://training.tpq.io

**Certificate in Computational Finance** | http://compfinance.tpq.io

**Derivatives Analytics with Python (Wiley Finance)** |
http://dawp.tpq.io

**Python for Finance (2nd ed., O'Reilly)** |
http://py4fi.tpq.io